# Module 3: 挑戰賽！

## 規則
- 2-3 人一組
- 從下方 5 個挑戰中選 1 個完成
- 時間：20 分鐘
- 最後每組 2 分鐘分享成果

```
挑戰難度指引：

Challenge A: 最多物件    ★☆☆☆☆  適合新手
Challenge B: 人流高峰    ★★☆☆☆  需要耐心
Challenge C: 動物獵人    ★★★☆☆  需要運氣
Challenge D: 多國文字    ★★★★☆  需要思考
Challenge E: 自由創作    ★★★★★  不設限！
```

---
## 環境設定

In [ ]:
!pip install ultralytics opencv-python-headless pillow easyocr -q

import cv2
import numpy as np
from PIL import Image
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import time
from collections import Counter
from ultralytics import YOLO

plt.rcParams['font.size'] = 12

def show_frame(frame, title='', size=(640, 360)):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    img = Image.fromarray(rgb).resize(size)
    if title:
        print(title)
    display(img)

def grab_frame(url, timeout=12):
    import subprocess
    # YouTube：用 yt-dlp 解析成可直接讀取的串流網址
    if 'youtube.com' in url or 'youtu.be' in url:
        try:
            real = subprocess.run(['yt-dlp', '-q', '-f', 'best[ext=mp4]/best', '-g', url],
                                  capture_output=True, text=True, timeout=45).stdout.strip().split('\n')[0]
            if real:
                url = real
        except Exception:
            pass
    # 國道即時影像是 MJPEG：用 curl 取一張 JPEG（cv2 對 MJPEG 不穩定）
    if 'abs2mjpg' in url:
        data = subprocess.run(['curl', '-s', '--max-time', '8', url], capture_output=True).stdout
        s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
        if s >= 0 and e >= 0:
            return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    else:
        cap = cv2.VideoCapture(url)
        cap.set(cv2.CAP_PROP_OPEN_TIMEOUT_MSEC, timeout * 1000)
        ret, frame = cap.read(); cap.release()
        if ret:
            return frame
    # 備援：台灣國道即時影像，保證有畫面
    print('來源連線失敗，改用國道即時影像備援')
    data = subprocess.run(['curl', '-s', '--max-time', '8',
        'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10001'], capture_output=True).stdout
    s = data.find(b'\xff\xd8'); e = data.find(b'\xff\xd9', s)
    if s >= 0 and e >= 0:
        return cv2.imdecode(np.frombuffer(data[s:e + 2], np.uint8), cv2.IMREAD_COLOR)
    raise ConnectionError('來源與備援皆失敗，請稍後再試')

# === 所有可用串流 ===
STREAMS = {
    # === 台灣國道即時影像（政府 CCTV，穩定、無金鑰，車流/交通最適合）===
    '國道1號 高架(車多)': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=11470',
    '國道1號 基隆端': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10001',
    '國道1號 0K+100': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=10010',
    '國道3號': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=30001',
    '國道2號 機場': 'https://cctvn.freeway.gov.tw/abs2mjpg/bmjpg?camera=20000',
    # === 也可用 YouTube 直播：grab_frame('https://www.youtube.com/watch?v=<直播ID>') ===
    # grab_frame 會自動用 yt-dlp 解析（直播或一般影片皆可）
}

model = YOLO('yolov8n.pt')
print(f'環境就緒！{len(STREAMS)} 個串流可用')
print()
for name in STREAMS:
    print(f'  {name}')

---
## Challenge A: 最多物件獵人 ★☆☆☆☆

**目標**：找到一個畫面，讓 YOLO 偵測出**最多不同類別**的物件。

**計分**：不同類別數量 = 你的分數

**提示**：購物台和旅遊台通常物件種類最多！

In [ ]:
# === Challenge A: 你的程式碼 ===
# 提示：試不同頻道，找到物件最豐富的畫面

best_score = 0
best_channel = ''
best_frame = None

# 試試這些頻道（你也可以加更多）
channels_to_try = ['國道1號 0K+100', '國道3號', '4K Travel', '國道1號 基隆端', '3sat DE']

for ch in channels_to_try:
    try:
        frame = grab_frame(STREAMS[ch])
        results = model(frame, verbose=False)
        classes = set(model.names[int(b.cls)] for b in results[0].boxes)
        score = len(classes)
        print(f'{ch}: {score} classes - {classes}')
        if score > best_score:
            best_score = score
            best_channel = ch
            best_frame = results[0].plot()
    except Exception:
        print(f'{ch}: offline')

# 顯示最佳結果
if best_frame is not None:
    print(f'\n=== Winner: {best_channel} with {best_score} different classes! ===')
    fig, ax = plt.subplots(figsize=(12, 7))
    ax.imshow(cv2.cvtColor(best_frame, cv2.COLOR_BGR2RGB))
    ax.set_title(f'Best: {best_channel} ({best_score} classes)', fontsize=16)
    ax.axis('off')
    plt.show()

---
## Challenge B: 人流高峰偵探 ★★☆☆☆

**目標**：追蹤一個新聞台 2 分鐘，找到**人數最多**的那一幀。

**計分**：最高人數 = 你的分數

**提示**：記者會或現場連線畫面通常人最多！

In [ ]:
# === Challenge B: 人流高峰追蹤 ===
channel = '國道1號 基隆端'  # 可換頻道
duration = 120  # 追蹤 2 分鐘
interval = 5    # 每 5 秒偵測一次

cap = cv2.VideoCapture(STREAMS[channel])
start = time.time()

max_people = 0
max_frame = None
timestamps = []
counts = []

print(f'追蹤 {channel}（{duration} 秒）...')

while time.time() - start < duration:
    ret, frame = cap.read()
    if not ret:
        break

    results = model(frame, classes=[0], conf=0.4, verbose=False)
    n = len(results[0].boxes)
    elapsed = time.time() - start

    timestamps.append(elapsed)
    counts.append(n)

    if n > max_people:
        max_people = n
        max_frame = results[0].plot()

    clear_output(wait=True)
    print(f'[{elapsed:.0f}s / {duration}s] People: {n} | Max so far: {max_people}')
    time.sleep(interval)

cap.release()

# 結果
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if max_frame is not None:
    axes[0].imshow(cv2.cvtColor(max_frame, cv2.COLOR_BGR2RGB))
    axes[0].set_title(f'Peak: {max_people} people!', fontsize=14)
    axes[0].axis('off')

axes[1].plot(timestamps, counts, 'o-', color='steelblue')
axes[1].fill_between(timestamps, counts, alpha=0.2, color='steelblue')
if timestamps:
    peak_idx = counts.index(max_people)
    axes[1].plot(timestamps[peak_idx], max_people, 'r*', markersize=20)
    axes[1].annotate(f'Peak: {max_people}', (timestamps[peak_idx], max_people),
                     fontsize=12, color='red')
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('People')
axes[1].set_title('People Count Over Time')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f'\n=== 你的分數: {max_people} 人 ===')

---
## Challenge C: 動物獵人 ★★★☆☆

**目標**：從野生動物頻道找到**最多種動物**。

**計分**：不同動物種類數 = 你的分數

**提示**：多試幾次，野生動物頻道的內容隨時在變！

YOLO 可辨識的動物：bird, cat, dog, horse, sheep, cow, elephant, bear, zebra, giraffe

In [ ]:
# === Challenge C: 動物獵人 ===
animal_classes = [14, 15, 16, 17, 18, 19, 20, 21, 22, 23]
animal_names = {i: model.names[i] for i in animal_classes}

wildlife_channels = ['國道2號 機場', 'Tanzania Safari', 'BBC Earth', '國道2號 機場']
found_animals = set()
best_frames = {}  # 每種動物保存一幀

# 每個頻道試 5 次
for ch in wildlife_channels:
    print(f'\n搜尋 {ch}...')
    for attempt in range(5):
        try:
            frame = grab_frame(STREAMS[ch])
            results = model(frame, classes=animal_classes, conf=0.3, verbose=False)
            for box in results[0].boxes:
                name = model.names[int(box.cls)]
                if name not in found_animals:
                    found_animals.add(name)
                    best_frames[name] = results[0].plot()
                    print(f'  發現新動物: {name}!')
            time.sleep(2)
        except Exception:
            print(f'  {ch}: offline')
            break

# 展示成果
print(f'\n=== 共發現 {len(found_animals)} 種動物: {found_animals} ===')

if best_frames:
    n = len(best_frames)
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6*cols, 5*rows))
    if n == 1:
        axes = [axes]
    else:
        axes = axes.flat if hasattr(axes, 'flat') else [axes]
    for ax, (name, frame) in zip(axes, best_frames.items()):
        ax.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Found: {name}', fontsize=14)
        ax.axis('off')
    for ax in list(axes)[n:]:
        ax.axis('off')
    plt.suptitle(f'Animal Collection: {len(found_animals)} species', fontsize=16)
    plt.tight_layout()
    plt.show()

---
## Challenge D: 多國文字收集家 ★★★★☆

**目標**：從不同國家的新聞台擷取文字，收集最多**不同語言**的文字。

**計分**：不同語言 / 國家的文字截圖數 = 你的分數

**提示**：日文台、韓文台、阿拉伯文台、中文台都試試！

In [ ]:
# === Challenge D: 多國文字 ===
import easyocr

# 每個語言需要載入不同的 OCR 模型
# 先用英文辨識所有台（英文幾乎都有）
reader_en = easyocr.Reader(['en'], gpu=True)
print('English OCR ready')

multilang_channels = {
    '國道1號 高架': '國道1號 高架(車多)',
    '國道1號 基隆端': '國道1號 基隆端',
    '國道1號 0K+100': '國道1號 0K+100',
    '國道3號': '國道3號',
}

collection = {}

for label, ch in multilang_channels.items():
    try:
        frame = grab_frame(STREAMS[ch])
        results = reader_en.readtext(frame)
        texts = [text for _, text, conf in results if conf > 0.4 and len(text) > 2]
        if texts:
            collection[label] = {
                'frame': frame,
                'texts': texts[:5],
            }
            print(f'{label}: {", ".join(texts[:3])}')
        else:
            print(f'{label}: (no text detected)')
    except Exception:
        print(f'{label}: offline')

# 展示收集成果
if collection:
    n = len(collection)
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    for ax, (label, data) in zip(axes.flat, collection.items()):
        ax.imshow(cv2.cvtColor(data['frame'], cv2.COLOR_BGR2RGB))
        ax.set_title(f'{label}\n{", ".join(data["texts"][:2])}', fontsize=10)
        ax.axis('off')
    for ax in axes.flat[n:]:
        ax.axis('off')
    plt.suptitle(f'Text Collection: {n} channels', fontsize=16)
    plt.tight_layout()
    plt.show()

print(f'\n=== 你的分數: {len(collection)} 個國家/語言 ===')

---
## Challenge E: 自由創作 ★★★★★

**目標**：結合你今天學到的技術，做出一個有創意的應用！

**靈感**：
- 「猜猜這是哪國的電視」— 用 OCR 辨識語言，推測國家
- 「新聞情緒分析」— 用人臉表情推測新聞正負面
- 「購物台商品計數器」— 統計購物台出現最多的物品
- 「全球人數比較」— 同時間不同國家新聞台的人數比較
- 「AI 動物觀察日記」— 連續觀察野生動物台，記錄出現了哪些動物

**計分**：創意 + 完成度 + 展示效果

In [ ]:
# === Challenge E: 範例 — 全球新聞台人數大比較 ===

global_news = {
    '國1高架': '國道1號 高架(車多)',
    '國1基隆': '國道1號 基隆端',
    '國1 0K+100': '國道1號 0K+100',
    '國3': '國道3號',
    '國2機場': '國道2號 機場',
}

comparison = {}

for region, ch in global_news.items():
    try:
        frame = grab_frame(STREAMS[ch])
        results = model(frame, classes=[0], conf=0.4, verbose=False)
        n = len(results[0].boxes)
        comparison[region] = {
            'count': n,
            'frame': results[0].plot(),
        }
        print(f'{region} ({ch}): {n} people')
    except Exception:
        print(f'{region} ({ch}): offline')

# 視覺化比較
if comparison:
    n = len(comparison)
    fig = plt.figure(figsize=(16, 10))

    # 上半部：各台畫面
    cols = min(n, 3)
    rows = (n + cols - 1) // cols
    for i, (region, data) in enumerate(comparison.items()):
        ax = fig.add_subplot(rows + 1, cols, i + 1)
        ax.imshow(cv2.cvtColor(data['frame'], cv2.COLOR_BGR2RGB))
        ax.set_title(f'{region}: {data["count"]} people')
        ax.axis('off')

    # 下半部：長條圖比較
    ax_bar = fig.add_subplot(rows + 1, 1, rows + 1)
    regions = list(comparison.keys())
    people_counts = [comparison[r]['count'] for r in regions]
    colors = plt.cm.Set2(np.linspace(0, 1, len(regions)))
    bars = ax_bar.bar(regions, people_counts, color=colors)
    ax_bar.set_ylabel('People Count')
    ax_bar.set_title('Global News: People Count Comparison')
    for bar, count in zip(bars, people_counts):
        ax_bar.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                    str(count), ha='center', fontsize=12, fontweight='bold')

    plt.tight_layout()
    plt.show()

In [ ]:
# === 你的自由創作空間 ===
# 在這裡寫你的程式碼！
#
# 可用工具：
#   grab_frame(STREAMS['頻道名'])  - 抓取一幀
#   model(frame)                   - YOLO 物件偵測
#   model(frame, classes=[0])      - 只偵測特定類別
#   reader_en.readtext(frame)      - OCR 文字辨識
#   plt.subplots()                 - 畫圖
#   show_frame(frame)              - 顯示影像




---
## 恭喜完成工作坊！

### 今天你學會了：

```
┌─────────────────────────────────────────┐
│  Module 0: 環境 + 串流 + 影像基礎       │
│  Module 1: YOLO 物件偵測 (6 種場景)      │
│  Module 2: 人數計算 + OCR + 交通監控     │
│  Module 3: 挑戰賽實戰                   │
└─────────────────────────────────────────┘
```

### 延伸學習：
- **YOLOv8 官方文檔**: https://docs.ultralytics.com
- **自訓練模型**: 用自己的資料集訓練 YOLO
- **串流來源**: https://github.com/iptv-org/iptv
- **EasyOCR**: https://github.com/JaidedAI/EasyOCR

### 進階方向：
- 物件追蹤 (Object Tracking) — 跨幀追蹤同一物件
- 姿態估計 (Pose Estimation) — 偵測人體骨架
- 語意分割 (Segmentation) — 像素級物件分割
- 自訂模型訓練 (Fine-tuning) — 辨識特定物件
- 部署到邊緣裝置 (Edge Deployment) — Raspberry Pi, Jetson